(where-mboostrs-stands-results-weaknesses-and-next-work)=
# mboostrs benchmark results and remaining work

This page summarises the benchmark evidence collected so far. It covers what
mboostrs does well, where it falls short, and which problems are being worked on next.
It is a status report, not a final evaluation. The studies differ in task, version,
split and machine, so they are not combined into one score. Each claim below names
the study it comes from.

The scorecard that frames the evidence is `docs/scorecard/scorecard.md`, version
`mboostrs-scorecard/2.1`. The qualitative status for each dimension is in
`docs/scorecard/status.md`. This page uses those documents and the study pages
linked below.

## Summary

- **Where it is competitive.** On the public data sets, mboostrs is close to the best
  library on negative binomial counts, on the spread of wages, and on the zinc surface
  when a tree term is added. In the claims study, its seven completed runs were the
  best calibrated of any library. Its additive
  core is fast: at 90,000 rows a gamboost fit takes 0.41 seconds, the fastest of the
  mboostrs variants and faster than CatBoost, XGBoost, LightGBM and pyGAM.
- **Where it falls short.** Boosted trees, CatBoost in particular, lead on most
  predictive comparisons. The default causal AIPW path gave absurd estimates in a
  controlled test on October 11; the October 16 rerun does not reproduce that. Its
  `smooth=False` arm still has about twice the error of EconML's linear arm. In the hostile workflows battery, the mboostrs analyst with the
  skill took about 40 minutes and 300 fits more than the arm without it, with no gain
  in correctness. Effect intervals undercover in the causal and KDD studies.
- **What is being fixed.** The issues are listed in the last section. The next work
  is on predictive validity in the autospec search, on effect intervals and the
  causal estimators, and on workflow cost and the missing measurements.

Two results need attention before anything else. The default AIPW failure of the
October 11 build does not reproduce in the October 16 rerun, but its `smooth=False`
arm is still less accurate than EconML's linear arm, as described below. The planned repair of the housing
validation, PR #540, was closed without merge, so the fix it contained is not on the
main branch.

(what-has-been-measured)=
## Measured studies

The table lists the studies behind this page. Each row names its question and its
headline result. Status is the study's own verdict: complete, partial, or a pilot that
did not pass its gate.

In [1]:
import polars as pl
from great_tables import GT

studies = pl.DataFrame(
    [
        {"Study": "Public data: count models", "Question": "How do count models score on held-out counts?",
         "Headline": "pyGAM 1.769, mboostrs NBinomial 1.790, mboostrs Poisson 1.793; CatBoost 1.875",
         "Status": "complete"},
        {"Study": "Public data: binary (Mroz)", "Question": "How do classifiers score on held-out labour-force rows?",
         "Headline": "With the wage variable: LightGBM 0.492, mboostrs Binomial 0.508; without it: mboostrs 0.630, logistic 0.627",
         "Status": "complete"},
        {"Study": "Public data: zinc (spatial)", "Question": "How much does each way of adding location help?",
         "Headline": "CatBoost 0.297, XGBoost 0.300, mboostrs with a tree on coordinates 0.304",
         "Status": "complete"},
        {"Study": "Public data: wage spread", "Question": "Does modelling the spread improve predictive distributions?",
         "Headline": "CatBoost uncertainty model 0.766; mboostrs GaussianLSS 0.777 vs 0.788 with constant spread",
         "Status": "complete"},
        {"Study": "Public data: scaling", "Question": "How does fit time grow with rows?",
         "Headline": "At 90,000 rows: gamboost 0.41 s, XGBoost 0.93 s, pyGAM 1.31 s, GaussianLSS 7.77 s",
         "Status": "complete"},
        {"Study": "Claims discovery (7 analysts)", "Question": "What does each library's analyst learn from one dataset?",
         "Headline": "statsmodels highest weighted score (3.48); mboostrs lowest in fully automated mode (2.95)",
         "Status": "complete; agent-driven"},
        {"Study": "Causal effects (job training)", "Question": "Can each library's analyst answer seven causal questions?",
         "Headline": "mboostrs weighted score 3.03 to 3.21 over four runs after library fixes",
         "Status": "partial; defects open"},
        {"Study": "Competitive shootout (tuned)", "Question": "How far behind the best competitor is each method on ten real datasets?",
         "Headline": "Median excess: CatBoost 0.1%, mboostrs hybrid 8.4%, autospec 38.3%, additive 46.5%",
         "Status": "complete"},
        {"Study": "Competitive refresh (small tier)", "Question": "Does the current autospec pipeline close the gap?",
         "Headline": "37 of 45 jobs complete; six housing failures from a validation defect; cancer +127% outer log loss",
         "Status": "partial"},
        {"Study": "Chooser on the shootout", "Question": "Does a capacity chooser beat the hand-set hybrid?",
         "Headline": "Median excess 12.7% against the hybrid's 7.9%; fixes cancer (+104% to +7.6%) and loses on bike panels",
         "Status": "partial; three causes open"},
        {"Study": "Stopping rules (#509, v2)", "Question": "Which rule for the number of boosting steps is safe?",
         "Headline": "Fold-local growth never worse in 28 cells; fold-local repeated-CV growth worse in 9, up to +107%",
         "Status": "complete; no defaults changed"},
        {"Study": "Joint regularization pilot", "Question": "Does an extra penalty remove whole terms?",
         "Headline": "Helps sparse and correlated signals, hurts dense and null; failed its gate; boosting stays default",
         "Status": "pilot, not adopted"},
        {"Study": "Paired estimation vs discovery (v1.1)", "Question": "Is a predictive gap estimation or discovery?",
         "Headline": "Most gaps unresolved at 8 replicates; on the interaction panel the starting specification costs 7.5% of variance",
         "Status": "complete at 8 replicates"},
        {"Study": "Hostile workflows (v4)", "Question": "Do the arms find planted structure and avoid traps?",
         "Headline": "All arms get the core verdicts; the skills arm costs 127 min and 686 fits against 87 min and 378 fits without skills",
         "Status": "complete; historical grading"},
        {"Study": "Causal estimator benchmark", "Question": "Where does mboostrs' causal gap come from?",
         "Headline": "Oct 11 build: default AIPW median error 294,000 dollars; current build 790; EconML linear 331",
         "Status": "complete; failure not reproduced on current build"},
        {"Study": "KDD scorecard study (planted truth)", "Question": "Scorecard v1 scores for mboostrs against a free-library analyst",
         "Headline": "mboostrs weighted 3.6 over eight scored dimensions; recovers all planted effects; two of four intervals miss",
         "Status": "complete; v1 card"},
        {"Study": "PM2.5 forecast competition", "Question": "How do methods forecast hourly PM2.5 on a time-shifted split?",
         "Headline": "LightGBM 0.621, CatBoost 0.618, mboostrs 0.660 log RMSE; 80% bands cover 0.74 overall, 0.62 to 0.80 by cell",
         "Status": "complete; uncertainty below nominal"},
    ]
)
(
    GT(studies)
    .tab_header(
        title="Benchmark studies to date and their headline results",
        subtitle="Each row is one study with its own task, version and split; the headlines are not directly comparable",
    )
    .tab_stubhead(label="Study")
    .tab_source_note(
        "Metrics: public data sets, held-out loss (lower is better); shootout, median percent excess over the best "
        "competitor (lower is better); causal and claims studies, weighted scorecard (higher is better). "
        "Sources: the study pages in this part of the book, and docs/scorecard/status.md."
    )
)

GT(_tbl_data=shape: (17, 4)
┌────────────────────────┬────────────────────────┬────────────────────────┬───────────────────────┐
│ Study                  ┆ Question               ┆ Headline               ┆ Status                │
│ ---                    ┆ ---                    ┆ ---                    ┆ ---                   │
│ str                    ┆ str                    ┆ str                    ┆ str                   │
╞════════════════════════╪════════════════════════╪════════════════════════╪═══════════════════════╡
│ Public data: count     ┆ How do count models    ┆ pyGAM 1.769, mboostrs  ┆ complete              │
│ models                 ┆ score on h…            ┆ NBinomia…              ┆                       │
│ Public data: binary    ┆ How do classifiers     ┆ With the wage          ┆ complete              │
│ (Mroz)                 ┆ score on he…           ┆ variable: LightG…      ┆                       │
│ Public data: zinc      ┆ How much does each way ┆ CatBoost 0.297,        ┆ complete              │
│ (spatial)              ┆ of addi…               ┆ XGBoost 0.300,…        ┆                       │
│ Public data: wage      ┆ Does modelling the     ┆ CatBoost uncertainty   ┆ complete              │
│ spread                 ┆ spread impr…           ┆ model 0.7…             ┆                       │
│ Public data: scaling   ┆ How does fit time grow ┆ At 90,000 rows:        ┆ complete              │
│                        ┆ with ro…               ┆ gamboost 0.41 …        ┆                       │
│ …                      ┆ …                      ┆ …                      ┆ …                     │
│ Paired estimation vs   ┆ Is a predictive gap    ┆ Most gaps unresolved   ┆ complete at 8         │
│ discovery…             ┆ estimation…            ┆ at 8 repl…             ┆ replicates            │
│ Hostile workflows (v4) ┆ Do the arms find       ┆ All arms get the core  ┆ complete; historical  │
│                        ┆ planted struc…         ┆ verdicts…              ┆ grading               │
│ Causal estimator       ┆ Where does mboostrs'   ┆ Oct 11 build: default  ┆ complete; failure not │
│ benchmark              ┆ causal ga…             ┆ AIPW med…              ┆ reproduc…             │
│ KDD scorecard study    ┆ Scorecard v1 scores    ┆ mboostrs weighted 3.6  ┆ complete; v1 card     │
│ (planted t…            ┆ for mboost…            ┆ over eig…              ┆                       │
│ PM2.5 forecast         ┆ How do methods         ┆ LightGBM 0.621,        ┆ complete; uncertainty │
│ competition            ┆ forecast hourly…       ┆ CatBoost 0.618…        ┆ below no…             │
└────────────────────────┴────────────────────────┴────────────────────────┴───────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10da658e0>, _boxhead=Boxhead([ColInfo(var='Study', type=<ColInfoTypeEnum.default: 1>, column_label='Study', column_align='left', column_width=None), ColInfo(var='Question', type=<ColInfoTypeEnum.default: 1>, column_label='Question', column_align='left', column_width=None), ColInfo(var='Headline', type=<ColInfoTypeEnum.default: 1>, column_label='Headline', column_align='left', column_width=None), ColInfo(var='Status', type=<ColInfoTypeEnum.default: 1>, column_label='Status', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10e04d6a0>, _spanners=Spanners([]), _heading=Heading(title='Benchmark studies to date and their headline results', subtitle='Each row is one study with its own task, version and split; the headlines are not directly comparable', preheader=None), _stubhead='Study', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10ea66ff0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10f259b50>, _source_notes=['Metrics: public data sets, held-out loss (lower is better); shootout, median percent excess over the best competitor (lower is better); causal and claims studies, weighted scorecard (higher is bett

## Capabilities

**The additive core is fast and calibrated where the data are smooth.** On the
wage-spread study, the mboostrs GaussianLSS model covers 91.3% of part-time workers' 90%
intervals, against 83.5% for the constant-spread model. Its spread ratio of 1.31 matches
the ratio in the residuals. On the scaling study it fits 90,000 rows in 0.41 seconds as
gamboost, and 7.8 seconds as the full distributional model.

**The count and spatial results are competitive.** On held-out bioChemists counts the
negative binomial mboostrs model is about 0.02 nats worse than the best model, pyGAM. On the
Meuse zinc data the mboostrs model with a tree on the coordinates is within 0.01 of
CatBoost.

**Discovery of an interaction in the controlled study.** In the planted-truth study
run by the scorecard (KDD study), mboostrs selected the four planted main effects and
the planted product term on all three datasets. Its false selections were limited to the
labelling question described in that study. In the paired-estimation study, the
discovery procedure did not lose ground to the supplied-specification arm on the
interaction panel; its median was slightly better, with a resolved negative gap of 1.5%
of the variance.

**Stopping rules are well behaved on the fold-local route.** Growing the number of boosting
steps within each fold was never worse than the fixed-budget rule in 28 cells. The comparisons use median loss changes within each panel/method/tier cell.
Its largest single-seed increase in loss was about 11%.

**Agent analysts can do the analysis.** In the claims study, the mboostrs analyst learned
the library from the package guide and delivered a pipeline that ran unseen on twelve
hidden draws. Six of seven analysts checked their pipeline on a permuted outcome; the
mboostrs pipeline was the one that did not, and the study records that gap.

**Recorded seeds, versions and hashes.** Each study records its protocol,
seeds, versions and hashes. The KDD study's two run directories reproduce byte for byte
from their own scripts, and the joint, chooser and stopping studies write machine-
readable artifacts.

## Weaknesses

**Boosted trees lead on most predictive comparisons.** On the ten real datasets of the
shootout, CatBoost is within 5% of the best competitor on 8, and its median excess is
0.1%. The mboostrs hybrid has a median excess of 8.4%. On the bike panels and the cancer
panel the gaps are larger. The chooser that was built to pick model capacity has a
median of 12.7%, worse than the hand-set hybrid across the thirteen panels, although it
fixes the cancer panel.

**Autospec is behind the hand-built hybrid.** Autospec's median excess is 38.3%. On the
joint boundary panel it reaches 314% above the best competitor, against 67% for the
hybrid, because its own search never selects the representation that fixes the panel.
In the current refresh, autospec completed 37 of 45 jobs on the small tier. Six housing
jobs failed because of a validation defect, and two superconductivity jobs timed out.

**The historical default AIPW arm failed on October 11.** In the causal estimator benchmark, the October 11 build's default AIPW
estimator returned ATE estimates ranging from about 16,500 to 19.1 million dollars
across ten effect draws, against a truth near 4,270. The default outcome model changed
in commit c011116, made after that run. In the October 16 rerun, the same `smooth=False` arm has a
median absolute error of about 790 dollars, against 331 for EconML's linear arm. The
no-clipping variant still has one draw with an error of about 59,700 dollars. The
coverage of 1.0 that the recorded default arm reported came from very large standard
errors, so it did not show calibration. Issues #420, #429 and #423 cover related AIPW
and effect-model problems.

The `mb_aipw_ate` arm in these figures was fitted with `smooth=False`, the default when it was recorded and when it was rerun on October 16. The current default is `smooth=True`, which fits the smooth arm, whose row is `mb_aipw_smooth`.

**Effect intervals undercover.** The causal rerun 6 reports 90% per-applicant coverage of
0.61 and an ATE interval coverage of 0.75. In the KDD study, two of four planted-effect
intervals missed the truth, and both lie below it. The shootout and the stopping study do
not report effect intervals, so this weakness is measured only in the causal and KDD
studies.

**Agent sessions cost more time and fits without better results.** The hostile workflows
battery found the same correct verdicts in all three arms. The skills arm used 127 minutes
and 686 fits; the arm without skills used 87 minutes and 378 fits. Its reported fit count
is self-reported, so the totals are a rough guide. On the joint-regularization pilot, the
median tuned-boosting fit was 0.009 to 0.057 seconds, so single fits are fast. The cost
is in the workflow, not in one fit.

**Uncertainty below nominal in the forecasting study.** In the PM2.5 competition the 80%
bands cover 0.74 of the held-out values overall, and between 0.62 and 0.80 by cell. The best
mboostrs cell covers 0.76. The bands widen with the predicted level, which is the
adaptive behaviour the scorecard asks for, but the coverage is not nominal in most cells.

**Several results are not in the main branch yet.** PR #540, which repairs the housing
validation, was closed without merge. The chooser results predate PR #502, and no
post-fix result is claimed for them.

**Two scoring decisions lower mboostrs' score in the claims study.** A null replicate
that the mboostrs pipeline did not finish counts as failing the null condition, which
gives mboostrs a D1 of 2. A failed run counts as zero coverage on its replicate, so its
prediction coverage over eight replicates is 0.79. Its seven completed runs were the best
calibrated of any library. Both decisions are stated in that study.

(where-the-scorecard-stands)=
## Scorecard status

The scorecard status page rates each dimension from the evidence. These are the
qualitative ratings from `docs/scorecard/status.md`, unchanged here:

| Dimension | Rating | Main gap |
|---|---|---|
| D1a Recovery under a prescribed specification | unmeasured; proxy only | no run isolates recovery of open structure |
| D1b Discovery from data alone | unmeasured | no no-formula battery yet (#511) |
| D2 Causal humility | mixed | valid adjustment sets, but no built-in doubly robust second stage |
| D3 Explanation quality | mixed | no consistent report-quality evidence across analyses |
| D4 Importance quantification | needs attention | interval coverage 0.61 to 0.75 in causal runs |
| D5 Iteration speed | unmeasured; cost proxies concerning | the 50k×20 session has not been run |
| D6 Robustness | needs attention | performance varies by regime; the joint pilot failed its gate |
| D7 Human friendliness | unmeasured | friction logged, no task-based study |
| D8 Agent friendliness | mixed | effective, but expensive in time and fits |
| D9 Uncertainty calibration | needs attention | PM2.5 bands below nominal |
| D10 Reproducibility | strong | one-command reproduction of a full end-user report not shown |
| D11 Predictive validity | mixed | competitive on some structures, large gaps on difficult panels |

The KDD study scored mboostrs on the earlier ten-dimension card, not on version 2.1.
Its weighted score was 3.6; version 2.1 requires reporting each dimension separately.

(what-is-being-fixed-next)=
## Open issues and priorities

The scorecard's current priorities are predictive validity, effect intervals and
calibration, and workflow efficiency. The table below lists the open issues behind them,
with the state each had on the date of this page. Items marked as closed are shown
because the status page cites them.

In [2]:
work = pl.DataFrame(
    [
        {"Priority": "1 Predictive validity", "Issue": "#493", "Work": "select_capacity: categorical-blind tree, tensor crash, single fixed budget", "State": "open"},
        {"Priority": "1 Predictive validity", "Issue": "#539", "Work": "autospec misses available joint capacity on non-coordinate panels", "State": "open"},
        {"Priority": "1 Predictive validity", "Issue": "#536", "Work": "pair-form paths capped by the additive optimum", "State": "open"},
        {"Priority": "1 Predictive validity", "Issue": "#500", "Work": "two-standard-error tolerance is inert on few time-ordered folds", "State": "open"},
        {"Priority": "1 Predictive validity", "Issue": "#541", "Work": "autospec cancer panel: +127% outer log loss despite low validation loss", "State": "open"},
        {"Priority": "1 Predictive validity", "Issue": "#538", "Work": "spatial residual-tree validation crashes on a single holdout", "State": "open; repair PR #540 closed unmerged"},
        {"Priority": "1 Predictive validity", "Issue": "#462", "Work": "separate sparsity, smoothness and optimisation in additive estimation", "State": "open; research"},
        {"Priority": "2 Effects and calibration", "Issue": "#443", "Work": "AIPW ATE intervals undercover under thin overlap", "State": "open; unsolved after three attempts"},
        {"Priority": "2 Effects and calibration", "Issue": "#420", "Work": "sparse-arm bias in aipw; refit extrapolation; effect_model auto choice", "State": "open"},
        {"Priority": "2 Effects and calibration", "Issue": "#429", "Work": "aipw nuisance control, messages, CV result shapes, IV estimator", "State": "open"},
        {"Priority": "2 Effects and calibration", "Issue": "#423", "Work": "effect_model R-learner blow-up; ci_test collinearity", "State": "open"},
        {"Priority": "2 Effects and calibration", "Issue": "#80", "Work": "repeated cross-fitting, cluster-robust variance, calibrated sensitivity", "State": "open"},
        {"Priority": "2 Effects and calibration", "Issue": "#136", "Work": "interval comparison with competitors calibrated under the same split", "State": "open"},
        {"Priority": "3 Workflow efficiency", "Issue": "#492", "Work": "do the agent skills earn their keep? Zero accuracy difference, higher cost", "State": "open"},
        {"Priority": "3 Workflow efficiency", "Issue": "#503", "Work": "D5: establish the analyst-session latency benchmark", "State": "open"},
        {"Priority": "Measurement", "Issue": "#511", "Work": "evaluation redesign: separate supplied-specification estimation from discovery", "State": "open"},
        {"Priority": "Causal friction", "Issue": "#484", "Work": "rerun 5 friction: iv_effect missing outcome, ci_test column naming", "State": "open"},
        {"Priority": "Causal friction", "Issue": "#487", "Work": "rerun 6 friction: linear default in aipw and iv_effect; no outcome-model helpers", "State": "open"},
    ]
)
(
    GT(work)
    .tab_header(
        title="Open work behind the current priorities",
        subtitle="Issues as of 2026-10-16; the priority column follows docs/scorecard/status.md",
    )
    .tab_stubhead(label="Priority")
    .tab_source_note("Issue states from the GitHub tracker on the date shown. PR #540 is listed as closed without merge.")
)

GT(_tbl_data=shape: (18, 4)
┌───────────────────────┬───────┬─────────────────────────────────┬───────┐
│ Priority              ┆ Issue ┆ Work                            ┆ State │
│ ---                   ┆ ---   ┆ ---                             ┆ ---   │
│ str                   ┆ str   ┆ str                             ┆ str   │
╞═══════════════════════╪═══════╪═════════════════════════════════╪═══════╡
│ 1 Predictive validity ┆ #493  ┆ select_capacity: categorical-b… ┆ open  │
│ 1 Predictive validity ┆ #539  ┆ autospec misses available join… ┆ open  │
│ 1 Predictive validity ┆ #536  ┆ pair-form paths capped by the … ┆ open  │
│ 1 Predictive validity ┆ #500  ┆ two-standard-error tolerance i… ┆ open  │
│ 1 Predictive validity ┆ #541  ┆ autospec cancer panel: +127% o… ┆ open  │
│ …                     ┆ …     ┆ …                               ┆ …     │
│ 3 Workflow efficiency ┆ #492  ┆ do the agent skills earn their… ┆ open  │
│ 3 Workflow efficiency ┆ #503  ┆ D5: establish the analyst-sess… ┆ open  │
│ Measurement           ┆ #511  ┆ evaluation redesign: separate … ┆ open  │
│ Causal friction       ┆ #484  ┆ rerun 5 friction: iv_effect mi… ┆ open  │
│ Causal friction       ┆ #487  ┆ rerun 6 friction: linear defau… ┆ open  │
└───────────────────────┴───────┴─────────────────────────────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x10ec6b650>, _boxhead=Boxhead([ColInfo(var='Priority', type=<ColInfoTypeEnum.default: 1>, column_label='Priority', column_align='left', column_width=None), ColInfo(var='Issue', type=<ColInfoTypeEnum.default: 1>, column_label='Issue', column_align='left', column_width=None), ColInfo(var='Work', type=<ColInfoTypeEnum.default: 1>, column_label='Work', column_align='left', column_width=None), ColInfo(var='State', type=<ColInfoTypeEnum.default: 1>, column_label='State', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10e244b00>, _spanners=Spanners([]), _heading=Heading(title='Open work behind the current priorities', subtitle='Issues as of 2026-10-16; the priority column follows docs/scorecard/status.md', preheader=None), _stubhead='Priority', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10e222f90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10e223bc0>, _source_notes=['Issue states from the GitHub tracker on the date shown. PR #540 is listed as closed without merge.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10e223380>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=Options

### Order of work

The order below follows the current priorities and the findings in this page. It is a
plan, not a commitment, and it changes as the measurements come in.

1. **Check the default AIPW path against the open AIPW issues.** The October 11 failure
   does not reproduce in the October 16 rerun, but its `smooth=False` arm still has
   about twice the error of EconML's linear arm, and one no-clipping draw is very large. The causal results depend
   on this estimator, so #420, #429 and #443 should be read against the rerun in the
   causal estimator benchmark.
2. **Restore the housing validation repair.** PR #540 was closed without merge, so the
   six housing failures in the refresh are still present on main. Either the repair
   goes back in, or #538 is answered another way.
3. **Predictive validity in autospec.** The chooser gaps (#493), the joint capacity gap
   (#539) and the cancer instability (#541) account for most of the predictive weakness
   on the shootout panels.
4. **Effect intervals.** Coverage is the weakness that most affects decisions, and the
   causal studies have not yet shown a fix (#443, #136).
5. **Workflow cost.** Reduce fits and wall-clock time for analysts without losing the
   checks that caught the defects (#492, #503).
6. **Measure what is not measured.** The no-formula discovery battery with the analyst inputs specified in #511, and the 50,000-row analyst session, are needed before any D1b or D5 claim.

(what-this-page-does-not-establish)=
## Limitations

- **No overall ranking.** Each study has its own task and split. The headlines are not
  averaged across studies.
- **No measurement of a full discovery report end to end.** D10 is strong in the named
  batteries but has not been shown for a complete user analysis.
- **Analyst variance is not measured.** Each library in the agent studies was run once,
  by one analyst. The variance over analysts is unknown.
- **Historical grades stay historical.** Results on older rubrics are kept with their
  studies and are not mapped onto version 2.1.